In [2]:
import pandas as pd
import requests
import time
import json
import os
import sys
import numpy as np
import re
from datetime import datetime
from difflib import SequenceMatcher
import logging

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("nba_data_processing.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("nba_data_processor")

# -----------------------------------------------------------------------------
# Data Scraping Functions
# -----------------------------------------------------------------------------

def fetch_play_by_play_json(game_id):
    """
    Fetch play-by-play data for a given game ID from the NBA API.
    
    Args:
        game_id (str): NBA game ID
        
    Returns:
        dict: JSON response or None if request failed
    """
    # Ensure game_id has proper format (with 00 prefix if needed)
   
    game_id = f'00{game_id}'
        
    url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{game_id}.json"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    }
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raise exception for HTTP errors
        
        logger.info(f"Successfully fetched data for game ID: {game_id}")
        return response.json()
        
    except requests.exceptions.RequestException as e:
        logger.error(f"Error fetching game {game_id}: {str(e)}")
        return None
    except json.JSONDecodeError:
        logger.error(f"Invalid JSON response for game {game_id}")
        return None

def fetch_multiple_games(game_ids, delay=0.5):
    """
    Fetch play-by-play data for multiple games.
    
    Args:
        game_ids (list): List of NBA game IDs
        delay (float): Delay between requests in seconds
        
    Returns:
        pandas.DataFrame: DataFrame containing all actions from all games
    """
    all_rows = []
    
    for game_id in game_ids:
        data = fetch_play_by_play_json(game_id)
        
        if data:
            actions = data.get('game', {}).get('actions', [])
            
            for action in actions:
                action['game_id'] = game_id  # Add the game ID as a column
                all_rows.append(action)
                
        time.sleep(delay)
    
    if not all_rows:
        logger.warning("No data was fetched for any of the provided game IDs")
        return pd.DataFrame()
        
    return pd.DataFrame(all_rows)

def pull_nba_stats_data(url):
    """
    Pull data from the NBA stats API.
    
    Args:
        url (str): NBA stats API URL
        
    Returns:
        pandas.DataFrame: DataFrame containing the response data
    """
    headers = {
        "Host": "stats.nba.com",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Referer": "https://stats.nba.com/",
        "Origin": "https://stats.nba.com",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        json_data = response.json()
        
        # Handle the video events format
        if 'resultSets' in json_data and isinstance(json_data['resultSets'], dict):
            if 'Meta' in json_data['resultSets'] and 'videoUrls' in json_data['resultSets']['Meta']:
                video_urls = json_data['resultSets']['Meta']['videoUrls']
                playlist = json_data['resultSets'].get('playlist', [])
                
                # Convert video URLs to dataframe
                video_df = pd.DataFrame(video_urls)
                
                # Convert playlist to dataframe
                playlist_df = pd.DataFrame(playlist)
                
                # Merge video and playlist data
                if not playlist_df.empty:
                    df = pd.concat([video_df, playlist_df], axis=1)
                else:
                    df = video_df
            else:
                # Fallback in case no video data is found
                df = pd.DataFrame()

        # Handle the original stats format
        elif 'resultSets' in json_data and isinstance(json_data['resultSets'], list):
            if len(json_data["resultSets"]) == 1:
                data = json_data["resultSets"][0]["rowSet"]
                columns = json_data["resultSets"][0]["headers"]
                df = pd.DataFrame.from_records(data, columns=columns)
            else:
                data = json_data["resultSets"][1]["rowSet"]
                columns = json_data["resultSets"][1]["headers"]["columnNames"]
                df = pd.DataFrame.from_records(data, columns=columns)
        else:
            # Empty dataframe if no recognizable format is found
            df = pd.DataFrame()
            
        time.sleep(1.2)  # Rate limiting
        return df
        
    except Exception as e:
        logger.error(f"Error pulling NBA stats data: {str(e)}")
        return pd.DataFrame()

# -----------------------------------------------------------------------------
# Data Processing Functions
# -----------------------------------------------------------------------------

def clock_to_seconds(clock_str):
    """
    Convert NBA clock format (PT12M00.00S) to seconds.
    
    Args:
        clock_str (str): Clock string in NBA format
        
    Returns:
        float: Time in seconds or None if invalid format
    """
    if pd.isna(clock_str) or not isinstance(clock_str, str):
        return None
        
    # Extract minutes and seconds using regex
    minutes_match = re.search(r'PT(\d+)M', clock_str)
    seconds_match = re.search(r'M(\d+\.\d+)S', clock_str)
    
    if not seconds_match:
        seconds_match = re.search(r'M(\d+)S', clock_str)
        
    minutes = int(minutes_match.group(1)) if minutes_match else 0
    seconds = float(seconds_match.group(1)) if seconds_match else 0
    
    return minutes * 60 + seconds

def calculate_description_similarity(desc1, desc2):
    """
    Calculate text similarity between two descriptions.
    
    Args:
        desc1 (str): First description
        desc2 (str): Second description
        
    Returns:
        float: Similarity score between 0 and 1
    """
    if pd.isna(desc1) or pd.isna(desc2):
        return 0
    
    # Normalize text for comparison
    desc1 = str(desc1).lower().strip()
    desc2 = str(desc2).lower().strip()
    
    # Use SequenceMatcher for string similarity
    return SequenceMatcher(None, desc1, desc2).ratio()

def extract_features(description):
    """
    Extract player names, action types, and points from a play description.
    
    Args:
        description (str): Play description text
        
    Returns:
        dict: Dictionary containing extracted features
    """
    if pd.isna(description):
        return {"player": "", "action": "", "points": ""}
    
    description = str(description).lower()
    
    # Extract player name (usually at the beginning)
    player_match = re.search(r'^([a-z][a-z\.\s]+)', description)
    player = player_match.group(1).strip() if player_match else ""
    
    # Extract action type
    actions = ["free throw", "jump shot", "layup", "dunk", "rebound", "assist", 
              "steal", "block", "turnover", "foul", "timeout", "substitution"]
    action = ""
    for act in actions:
        if act in description:
            action = act
            break
    
    # Extract points if present
    points_match = re.search(r'\((\d+)\s*PTS\)', description)
    points = points_match.group(1) if points_match else ""
    
    return {"player": player, "action": action, "points": points}

def prepare_play_by_play_data(pbp_df):
    """
    Prepare play-by-play data by calculating game seconds and sorting.
    
    Args:
        pbp_df (pandas.DataFrame): Play-by-play DataFrame
        
    Returns:
        pandas.DataFrame: Prepared DataFrame with game seconds
    """
    # Make a copy to avoid modifying the original
    df = pbp_df.copy()
    
    # Calculate game seconds for each action
    df['clock_seconds'] = df['clock'].apply(clock_to_seconds)
    df['period_start_seconds'] = (df['period'].astype(int) - 1) * 720
    df['seconds_into_period'] = 720 - df['clock_seconds']
    df['game_seconds'] = df['period_start_seconds'] + df['seconds_into_period']

    # Sort by period and game_seconds
    df.sort_values(by=['period', 'game_seconds'], inplace=True)
    
    return df

def map_action_numbers(old_df, new_df, description_weight=0.7, time_weight=0.3, debug=False):
    """
    Map actionNumber from new_df to the appropriate rows in old_df based on time ranges and text similarity.
    
    Args:
        old_df (pandas.DataFrame): The old dataset with start_seconds, end_seconds and DESCRIPTION columns
        new_df (pandas.DataFrame): The new dataset with actionNumber, period, clock, and description columns
        description_weight (float): Weight given to description similarity (between 0 and 1)
        time_weight (float): Weight given to time proximity (between 0 and 1)
        debug (bool): Whether to add debug columns to the output
        
    Returns:
        pandas.DataFrame: A copy of old_df with an 'actionNumber' column and debug info if requested
    """
    logger.info(f"Starting action mapping with {len(old_df)} old events and {len(new_df)} new events")
    logger.info(f"Description weight: {description_weight}, Time weight: {time_weight}")
    
    # Make a copy of the old_df to avoid modifying the original
    result_df = old_df.copy()
    
    # Initialize the actionNumber column with NaN
    result_df['actionNumber'] = np.nan
    
    # Convert data types to ensure proper comparison
    result_df['PERIOD'] = result_df['PERIOD'].astype(int)
    result_df['start_seconds'] = result_df['start_seconds'].astype(float)
    result_df['end_seconds'] = result_df['end_seconds'].astype(float)
    
    # Prepare the new dataframe
    new_df = prepare_play_by_play_data(new_df)
    
    # Group by period for faster access
    new_df_by_period = {period: group for period, group in new_df.groupby('period')}
    
    # Debug columns
    if debug:
        result_df['debug_potential_matches'] = 0
        result_df['debug_best_score'] = 0.0
        result_df['debug_time_proximity'] = 0.0
        result_df['debug_desc_similarity'] = 0.0
        result_df['debug_matched_description'] = ""
        result_df['debug_time_diff'] = np.nan
    
    # Function to find the action number with debug info
    def find_action_number(row):
        period = row['PERIOD']
        start_time = row['start_seconds']
        end_time = row['end_seconds']
        old_description = row['DESCRIPTION'] if 'DESCRIPTION' in row else ""
        
        # Debug dictionary to store information
        debug_info = {
            'potential_matches': 0,
            'best_score': -1,
            'time_proximity': 0,
            'desc_similarity': 0,
            'matched_description': "",
            'time_diff': np.nan
        }
        
        # Check if we have data for this period
        if period not in new_df_by_period:
            logger.warning(f"No data found for period {period}")
            return (np.nan, debug_info) if debug else np.nan
        
        period_data = new_df_by_period[period]
        
        # Find actions that fall within or near the time range
        time_window = 1  # seconds
        
        potential_matches = period_data[
            (period_data['game_seconds'] >= (start_time - time_window)) & 
            (period_data['game_seconds'] <= (end_time + time_window))
        ]

        debug_info['potential_matches'] = len(potential_matches)
        
        if potential_matches.empty:
            return (np.nan, debug_info) if debug else np.nan
        
        # If only one match, return it
        if len(potential_matches) == 1:
            best_match = potential_matches['actionNumber'].iloc[0]
            
            # Still populate debug info for single matches
            if debug:
                if 'description' in potential_matches.columns:
                    debug_info['matched_description'] = potential_matches['description'].iloc[0]
                    debug_info['desc_similarity'] = calculate_description_similarity(
                        old_description, potential_matches['description'].iloc[0]
                    )
                
                debug_info['time_diff'] = abs(
                    potential_matches['game_seconds'].iloc[0] - (start_time + end_time)/2
                )
                debug_info['time_proximity'] = max(
                    0, 1 - (debug_info['time_diff'] / (time_window * 2))
                )
                debug_info['best_score'] = 1.0  # Perfect match (only one candidate)
                
            return (best_match, debug_info) if debug else best_match
        
        # Calculate similarities and combine with time proximity
        best_match = None
        best_score = -1

        old_features = extract_features(old_description)
        
        for idx, match in potential_matches.iterrows():
            # Time proximity score (1 for exact match, decreases with distance)
            time_diff = abs(match['game_seconds'] - (start_time + end_time)/2)
            time_proximity = max(0, 1 - (time_diff / (time_window * 2)))
            
            # Description similarity
            desc_similarity = 0
            if 'description' in match and not pd.isna(match['description']):
                desc_similarity = calculate_description_similarity(old_description, match['description'])
                
                # Feature-based similarity
                new_features = extract_features(match['description'])
                feature_similarity = 0
                if old_features["player"] and old_features["player"] == new_features["player"]:
                    feature_similarity += 0.5
                if old_features["action"] and old_features["action"] == new_features["action"]:
                    feature_similarity += 0.3
                if old_features["points"] and old_features["points"] == new_features["points"]:
                    feature_similarity += 0.2
                
                desc_similarity = max(desc_similarity, feature_similarity)
            
            # Combined score
            combined_score = (description_weight * desc_similarity) + (time_weight * time_proximity)
            
            if combined_score > best_score:
                best_score = combined_score
                best_match = match['actionNumber']
                
                # Update debug info
                if debug:
                    debug_info['best_score'] = best_score
                    debug_info['time_proximity'] = time_proximity
                    debug_info['desc_similarity'] = desc_similarity
                    debug_info['matched_description'] = match.get('description', "")
                    debug_info['time_diff'] = time_diff
        
        return (best_match, debug_info) if debug else best_match
    
    # Apply the function to each row in the old dataset
    if debug:
        # For debug mode, we get back a tuple of (action_number, debug_info)
        results = result_df.apply(find_action_number, axis=1)
        
        # Split the results into action numbers and debug info
        result_df['actionNumber'] = [r[0] for r in results]
        
        # Add debug columns
        for i, r in enumerate(results):
            debug_info = r[1]
            result_df.at[i, 'debug_potential_matches'] = debug_info['potential_matches']
            result_df.at[i, 'debug_best_score'] = debug_info['best_score']
            result_df.at[i, 'debug_time_proximity'] = debug_info['time_proximity']
            result_df.at[i, 'debug_desc_similarity'] = debug_info['desc_similarity']
            result_df.at[i, 'debug_matched_description'] = debug_info['matched_description']
            result_df.at[i, 'debug_time_diff'] = debug_info['time_diff']
    else:
        result_df['actionNumber'] = result_df.apply(find_action_number, axis=1)
    
    # Add a confidence score based on description similarity
    if 'DESCRIPTION' in result_df.columns and 'description' in new_df.columns:
        def calculate_confidence(row):
            if pd.isna(row['actionNumber']):
                return 0
                
            action_data = new_df[new_df['actionNumber'] == row['actionNumber']]
            if action_data.empty:
                return 0
                
            return calculate_description_similarity(
                row['DESCRIPTION'], 
                action_data['description'].iloc[0]
            )
            
        result_df['match_confidence'] = result_df.apply(calculate_confidence, axis=1)
    
    # Log statistics
    total_rows = len(result_df)
    mapped_rows = result_df['actionNumber'].notna().sum()
    mapping_percentage = (mapped_rows / total_rows) * 100
    
    logger.info(f"Total rows in old dataset: {total_rows}")
    logger.info(f"Successfully mapped rows: {mapped_rows} ({mapping_percentage:.2f}%)")
    
    # If debug mode, log some examples of both successful and failed matches
    if debug and not result_df.empty:
        # Examples of successful matches
        successful = result_df[result_df['actionNumber'].notna()].sort_values('debug_best_score', ascending=False)
        if not successful.empty:
            top_3 = successful.head(3)
            logger.info("TOP 3 SUCCESSFUL MATCHES:")
            for i, row in top_3.iterrows():
                logger.info(f"Old: '{row['DESCRIPTION']}' -> New: '{row['debug_matched_description']}'")
                logger.info(f"  Score: {row['debug_best_score']:.3f} (Time: {row['debug_time_proximity']:.3f}, Desc: {row['debug_desc_similarity']:.3f})")
        
        # Examples of failed matches
        failed = result_df[result_df['actionNumber'].isna()]
        if not failed.empty:
            sample_3 = failed.sample(min(3, len(failed)))
            logger.info("SAMPLE OF FAILED MATCHES:")
            for i, row in sample_3.iterrows():
                logger.info(f"Failed to match: '{row['DESCRIPTION']}'")
                logger.info(f"  Potential matches: {row['debug_potential_matches']}")
    
    return result_df

# -----------------------------------------------------------------------------
# Main Processing Functions
# -----------------------------------------------------------------------------

def process_missing_games(team_data_path, missing_data_path, output_path=None, debug=True):
    """
    Process team data to find missing URLs and fetch play-by-play data for those games.
    
    Args:
        team_data_path (str): Path to the team data CSV file
        missing_data_path (str): Path to the CSV containing game IDs with missing data
        output_path (str, optional): Path to save the mapped results
        debug (bool): Whether to add debug columns to the output
        
    Returns:
        pandas.DataFrame: Merged data with mapped action numbers
    """
    logger.info(f"Processing team data from {team_data_path}")
    
    # Read team data
    all_df = pd.read_csv(team_data_path)
    missing_frame = pd.read_csv(missing_data_path)
    
    # Identify the rows to exclude
    mask = all_df['GAMEID'].isin(missing_frame['game_id'].unique())
    
    # Set the 'URL' column to None for matching GAMEIDs
    all_df.loc[mask, 'URL'] = None
    
    # Identify GAME_IDs with at least one non-NaN URL
    valid_games = all_df[all_df['URL'].notna()]['GAMEID'].unique()
    
    # Filter out GAME_IDs without any non-NaN URLs
    missing_url_games = all_df[~all_df['GAMEID'].isin(valid_games)]['GAMEID'].unique()
    
    logger.info(f"Found {len(missing_url_games)} games with missing URLs:")
    logger.info(missing_url_games)
    
    # Filter data for missing games
    missing_data = all_df[all_df['GAMEID'].isin(missing_url_games)].copy()
    
    # Fetch play-by-play data for missing games
    pbp_data = fetch_multiple_games(missing_url_games)
    
    if pbp_data.empty:
        logger.warning("No play-by-play data could be fetched. Returning original data.")
        return missing_data
    
    # Clean up data for merging
    missing_data['GAMEID'] = '00' + missing_data['GAMEID'].astype(str)
    missing_data.sort_values(by='GAMEDATE', inplace=True)
    
    pbp_data.sort_values(by='timeActual', inplace=True)
    pbp_data.dropna(subset=['teamId'], inplace=True)
    pbp_data['teamId'] = pbp_data['teamId'].astype(int)
    
    # Filter for the team's data
    team_id = missing_data['TEAM_ID'].iloc[0]
    pbp_data_team = pbp_data[pbp_data['teamId'] == team_id]
    
    # Map action numbers
    result_df = map_action_numbers(missing_data, pbp_data_team, debug=debug)
    
    # Save results if output path is provided
    if output_path:
        result_df.to_csv(output_path, index=False)
        logger.info(f"Mapping results saved to {output_path}")
    
    return result_df

def test_mapping_parameters(old_df, new_df, param_grid=None):
    """
    Test different parameter combinations for action mapping and return the best.
    
    Args:
        old_df (pandas.DataFrame): The old dataset with start_seconds, end_seconds and DESCRIPTION columns
        new_df (pandas.DataFrame): The new dataset with actionNumber, period, clock, and description columns
        param_grid (dict, optional): Dictionary of parameter combinations to test
        
    Returns:
        tuple: (best_params, best_result_df, performance_metrics)
    """
    if param_grid is None:
        param_grid = {
            'description_weight': [0.5, 0.6, 0.7, 0.8],
            'time_weight': [0.5, 0.4, 0.3, 0.2]
        }
    
    logger.info("Testing different parameter combinations for action mapping")
    
    best_mapping_percentage = 0
    best_params = {}
    best_result_df = None
    performance_metrics = []
    
    for desc_weight in param_grid['description_weight']:
        for time_weight in param_grid['time_weight']:
            # Normalize weights
            total = desc_weight + time_weight
            norm_desc_weight = desc_weight / total
            norm_time_weight = time_weight / total
            
            logger.info(f"Testing with description_weight={norm_desc_weight:.2f}, time_weight={norm_time_weight:.2f}")
            
            # Map actions with current parameters
            result_df = map_action_numbers(
                old_df, 
                new_df, 
                description_weight=norm_desc_weight,
                time_weight=norm_time_weight,
                debug=True
            )
            
            # Calculate performance metrics
            total_rows = len(result_df)
            mapped_rows = result_df['actionNumber'].notna().sum()
            mapping_percentage = (mapped_rows / total_rows) * 100
            
            # Calculate average match score for successful matches
            avg_score = result_df.loc[result_df['actionNumber'].notna(), 'debug_best_score'].mean()
            
            metrics = {
                'description_weight': norm_desc_weight,
                'time_weight': norm_time_weight,
                'mapping_percentage': mapping_percentage,
                'mapped_rows': mapped_rows,
                'total_rows': total_rows,
                'avg_match_score': avg_score
            }
            
            performance_metrics.append(metrics)
            
            logger.info(f"Results: {mapping_percentage:.2f}% mapped, avg score: {avg_score:.4f}")
            
            # Check if this is the best so far
            if mapping_percentage > best_mapping_percentage or (
                mapping_percentage == best_mapping_percentage and avg_score > 
                performance_metrics[performance_metrics.index(best_params)]['avg_match_score']
            ):
                best_mapping_percentage = mapping_percentage
                best_params = metrics
                best_result_df = result_df
    
    logger.info(f"Best parameters: description_weight={best_params['description_weight']:.2f}, " + 
                f"time_weight={best_params['time_weight']:.2f}")
    logger.info(f"Best mapping percentage: {best_params['mapping_percentage']:.2f}%")
    
    return best_params, best_result_df, performance_metrics

# -----------------------------------------------------------------------------
# Main Execution Flow (Example)
# -----------------------------------------------------------------------------

if __name__ == "__main__":
    # Example usage
    
    # Define paths for data files
    team_data_dir = "2025"
    missing_data_path = "all_missing.csv"
    output_dir = "results"
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # List of teams to process
    teams = ['ATL', 'MIA']  # Add more teams as needed
    
    # Process each team
    for team in teams:
        team_data_path = f"{team_data_dir}/{team}_2025_clips_with_players.csv"
        output_path = f"{output_dir}/{team}_mapped_results.csv"
        
        logger.info(f"Processing team: {team}")
        
        if not os.path.exists(team_data_path):
            logger.error(f"Team data file not found: {team_data_path}")
            continue
        
        # Process missing games for this team
        result_df = process_missing_games(
            team_data_path=team_data_path,
            missing_data_path=missing_data_path,
            output_path=output_path,
            debug=True
        )
        
        logger.info(f"Completed processing for {team}")

2025-03-30 17:28:09,242 - nba_data_processor - INFO - Processing team: ATL


2025-03-30 17:28:09,243 - nba_data_processor - INFO - Processing team data from 2025/ATL_2025_clips_with_players.csv
2025-03-30 17:28:09,886 - nba_data_processor - INFO - Found 10 games with missing URLs:
2025-03-30 17:28:09,888 - nba_data_processor - INFO - [22400239 22400719 22400945 22400960 22400978 22400993 22401025 22401031
 22401049 22401062]
2025-03-30 17:28:10,223 - nba_data_processor - INFO - Successfully fetched data for game ID: 0022400239
2025-03-30 17:28:11,043 - nba_data_processor - INFO - Successfully fetched data for game ID: 0022400719
2025-03-30 17:28:11,867 - nba_data_processor - INFO - Successfully fetched data for game ID: 0022400945
2025-03-30 17:28:12,682 - nba_data_processor - INFO - Successfully fetched data for game ID: 0022400960
2025-03-30 17:28:13,505 - nba_data_processor - INFO - Successfully fetched data for game ID: 0022400978
2025-03-30 17:28:14,366 - nba_data_processor - INFO - Successfully fetched data for game ID: 0022400993
2025-03-30 17:28:15,189 